# Aggregate Metrics 

This notebook reads the `ss_metrics.csv` file produced by Safety System excecutions and computes the summary metrics, then displays and saves the results.

Requirements:\n
- Python 3 with `pandas` installed. Install with `pip install pandas` if needed.

Usage:\n
- Put your `ss_metrics.csv` next to this notebook or pass a path to the variable `CSV_PATH` in the first code cell.
- Run the notebook cells in order.

In [1]:
# Configuration: set CSV_PATH to your metrics CSV file (default: ss_metrics.csv)
CSV_PATH = "ss_metrics.csv"

In [2]:
# Imports
import re
import math
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
# helper to parse first timestamp from a field (handles numbers, list-like strings like [123, 456])
def parse_first_timestamp(val):
    if pd.isna(val):
        return None
    s = str(val)
    m = re.findall(r'([0-9]{9,})', s)
    if m:
        try:
            return int(m[0])
        except Exception:
            return None
    try:
        return int(s)
    except Exception:
        return None

def truthy_from_cols(df, keys):
    # Returns boolean Series where any of the provided keys indicates truth
    out = pd.Series(False, index=df.index)
    for k in keys:
        if k in df.columns:
            s = df[k].astype(str).str.lower()
            bool_str = s.isin(['1','true','t'])
            numeric = pd.to_numeric(df[k], errors='coerce')==1
            out = out | bool_str | numeric.fillna(False)
    return out

In [4]:
# Read CSV
p = Path(CSV_PATH)
if not p.exists():
    print(f'CSV not found at {CSV_PATH} — update CSV_PATH variable and re-run')
else:
    df = pd.read_csv(p)
    print('Loaded', len(df), 'rows from', p)

Loaded 27 rows from ss_metrics.csv


In [5]:
# Compute M8-M14
def compute_metrics(df):
    total = len(df)
    # M8: reached safe terminal condition: enforced_stop OR safe_halt_active
    enforced = truthy_from_cols(df, ['enforced_stop', 'enforced_stop_timestamp'])
    safe_active = truthy_from_cols(df, ['safe_halt_active'])
    geofence = truthy_from_cols(df, ['geofence_violation'])

    m8_reached_safe_fraction = ((enforced | safe_active).sum() / total) if total else None
    m10_safe_active_fraction = (safe_active.sum() / total) if total else None
    m11_geofence_fraction = (geofence.sum() / total) if total else None

    # M12: global minimum distance across runs (min of min_distance column)
    min_distance = None
    for col in ('min_distance', 'min_distance_observed'):
        if col in df.columns:
            try:
                vals = pd.to_numeric(df[col], errors='coerce').dropna()
                if not vals.empty:
                    cur = vals.min()
                    if min_distance is None or cur < min_distance:
                        min_distance = float(cur)
            except Exception:
                pass

    # M13 normal: safe_halt_timestamp - safe_halt_request_timestamp (average over runs that have both)
    normal_diffs = []
    enforced_diffs = []
    geofence_latencies = []
    for idx, row in df.iterrows():
        sr_req = parse_first_timestamp(row.get('safe_halt_request_timestamp') or row.get('safe_halt_request_timestamps') or row.get('safe_halt_request'))
        sr_ts = parse_first_timestamp(row.get('safe_halt_timestamp'))
        en_ts = parse_first_timestamp(row.get('enforced_stop_timestamp'))
        gv_ts = parse_first_timestamp(row.get('geofence_violation_timestamp'))
        if sr_req is not None and sr_ts is not None:
            normal_diffs.append(sr_ts - sr_req)
        if sr_req is not None and en_ts is not None:
            enforced_diffs.append(en_ts - sr_req)
        if gv_ts is not None and en_ts is not None:
            geofence_latencies.append(en_ts - gv_ts)

    def avg(lst):
        return (sum(lst) / len(lst)) if lst else None

    metrics = {
        'Total runs': total,
        'M8 – Probability of reaching a safe terminal condition': m8_reached_safe_fraction,
        'M10 – Frequency of Safety System intervention': m10_safe_active_fraction,
        'M11 – Frequency of minimum-distance violations': m11_geofence_fraction,
        'M12 – Minimum distance to restricted boundary': min_distance,
        'M13 – Safe halt response time milliseconds (halt observed normally)': avg(normal_diffs),
        'M13 – Safe halt response time milliseconds (enforced by SS)': avg(enforced_diffs),
        'M14 – Geofence enforcement latency milliseconds': avg(geofence_latencies),
    }
    # Add detailed per-run columns (parsed timestamps + diffs)
    detail = df.copy()
    detail['safe_halt_request_ts_parsed'] = detail['safe_halt_request_timestamp'].apply(lambda v: parse_first_timestamp(v) if 'safe_halt_request_timestamp' in detail.columns else None) if 'safe_halt_request_timestamp' in detail.columns else None
    # fallback columns handling
    if 'safe_halt_request_timestamps' in detail.columns:
        detail['safe_halt_request_ts_parsed'] = detail.apply(lambda r: parse_first_timestamp(r.get('safe_halt_request_timestamp') or r.get('safe_halt_request_timestamps') or r.get('safe_halt_request')), axis=1)
    else:
        detail['safe_halt_request_ts_parsed'] = detail.apply(lambda r: parse_first_timestamp(r.get('safe_halt_request_timestamp') or r.get('safe_halt_request')), axis=1)
    detail['safe_halt_ts_parsed'] = detail['safe_halt_timestamp'].apply(parse_first_timestamp) if 'safe_halt_timestamp' in detail.columns else None
    detail['enforced_stop_ts_parsed'] = detail['enforced_stop_timestamp'].apply(parse_first_timestamp) if 'enforced_stop_timestamp' in detail.columns else None
    detail['geofence_violation_ts_parsed'] = detail['geofence_violation_timestamp'].apply(parse_first_timestamp) if 'geofence_violation_timestamp' in detail.columns else None
    detail['safe_halt_response_normal_ms'] = detail.apply(lambda r: (r['safe_halt_ts_parsed'] - r['safe_halt_request_ts_parsed']) if pd.notna(r.get('safe_halt_ts_parsed')) and pd.notna(r.get('safe_halt_request_ts_parsed')) else None, axis=1)
    detail['safe_halt_response_enforced_ms'] = detail.apply(lambda r: (r['enforced_stop_ts_parsed'] - r['safe_halt_request_ts_parsed']) if pd.notna(r.get('enforced_stop_ts_parsed')) and pd.notna(r.get('safe_halt_request_ts_parsed')) else None, axis=1)
    detail['geofence_enforcement_latency_ms'] = detail.apply(lambda r: (r['enforced_stop_ts_parsed'] - r['geofence_violation_ts_parsed']) if pd.notna(r.get('enforced_stop_ts_parsed')) and pd.notna(r.get('geofence_violation_ts_parsed')) else None, axis=1)

    # Additional aggregated metrics for summary tables 4-7
    sums = {}
    for col in [
        'geofence_count_monitoring', 'monitoring_time_ms',
        'geofence_count_waitforhalt', 'waitforhalt_time_ms',
        'geofence_count_safehaltactive', 'safehaltactive_time_ms',
        'haltObserved_in_wait', 'tick_in_wait', 'waitforhalt_entry_count',
        'move_in_safehalt', 'safehaltactive_entry_count', 'safe_halt_request_count'
    ]:
        if col in df.columns:
            sums[col] = int(pd.to_numeric(df[col], errors='coerce').fillna(0).sum())

    request_timestamps = []
    for idx, row in df.iterrows():
        req = parse_first_timestamp(row.get('safe_halt_request_timestamp') or row.get('safe_halt_request_timestamps') or row.get('safe_halt_request'))
        if req is not None:
            request_timestamps.append(req)
    if request_timestamps:
        sums['safe_halt_request_timestamp_min'] = int(min(request_timestamps))
        sums['safe_halt_request_timestamp_max'] = int(max(request_timestamps))

    return metrics, detail, sums

# Compute and show
metrics, detail, sums = compute_metrics(df)
import pprint
pprint.pprint(metrics)

# Display summary as a small DataFrame
summary_df = pd.DataFrame(list(metrics.items()), columns=['metric','value'])
display(summary_df)

# Build Table 4–7 summaries

table4 = pd.DataFrame([
    ['Monitoring', int(sums.get('geofence_count_monitoring', 0)), int(sums.get('monitoring_time_ms', 0))],
    ['WaitForHalt', int(sums.get('geofence_count_waitforhalt', 0)), int(sums.get('waitforhalt_time_ms', 0))],
    ['SafeHaltActive', int(sums.get('geofence_count_safehaltactive', 0)), int(sums.get('safehaltactive_time_ms', 0))],
], columns=['State','count geofenceViolation','total time ms'])

table5 = pd.DataFrame([
    ['WaitForHalt', int(sums.get('haltObserved_in_wait', 0)), int(sums.get('tick_in_wait', 0)), int(sums.get('waitforhalt_entry_count', 0))],
], columns=['State','count haltObserved','count tick','total entries'])

table6 = pd.DataFrame([
    ['SafeHaltActive', int(sums.get('move_in_safehalt', 0)), int(sums.get('safehaltactive_entry_count', 0))],
], columns=['State','count move','total entries'])


print('\nTable 4 — Geofence Monitoring (SS_SafeHaltController)')
display(table4)
print('\nTable 5 — WaitForHalt Outcomes (SS_SafeHaltController)')
display(table5)
print('\nTable 6 — SafeHaltActive "move" outcomes (SS_SafeHaltController)')
display(table6)


# Save outputs
summary_out = Path('metrics_summary.csv')
detail_out = Path('metrics_detailed.csv')
summary_df.to_csv(summary_out, index=False)
detail.to_csv(detail_out, index=False)
table4.to_csv(Path('metrics_table4.csv'), index=False)
table5.to_csv(Path('metrics_table5.csv'), index=False)
table6.to_csv(Path('metrics_table6.csv'), index=False)

print('Saved summary to', summary_out, 'and detail to', detail_out)
print('Saved table files: metrics_table4.csv, metrics_table5.csv, metrics_table6.csv, metrics_table7.csv')

{'M10 – Frequency of Safety System intervention': np.float64(0.0),
 'M11 – Frequency of minimum-distance violations': np.float64(0.037037037037037035),
 'M12 – Minimum distance to restricted boundary': 0.4017219245433807,
 'M13 – Safe halt response time milliseconds (enforced by SS)': None,
 'M13 – Safe halt response time milliseconds (halt observed normally)': None,
 'M14 – Geofence enforcement latency milliseconds': 32.0,
 'M8 – Probability of reaching a safe terminal condition': np.float64(0.037037037037037035),
 'Total runs': 27}


,metric,value
0,Total runs,27.000000
1,M8 – Probability of reaching a safe terminal c...,0.037037
2,M10 – Frequency of Safety System intervention,0.000000
3,M11 – Frequency of minimum-distance violations,0.037037
4,M12 – Minimum distance to restricted boundary,0.401722
5,M13 – Safe halt response time milliseconds (ha...,NaN
6,M13 – Safe halt response time milliseconds (en...,NaN
7,M14 – Geofence enforcement latency milliseconds,32.000000



Table 4 — Geofence Monitoring (SS_SafeHaltController)


,State,count geofenceViolation,total time ms
0,Monitoring,1,2532510
1,WaitForHalt,0,0
2,SafeHaltActive,0,0



Table 5 — WaitForHalt Outcomes (SS_SafeHaltController)


,State,count haltObserved,count tick,total entries
0,WaitForHalt,0,0,0



Table 6 — SafeHaltActive "move" outcomes (SS_SafeHaltController)


,State,count move,total entries
0,SafeHaltActive,0,0


Saved summary to metrics_summary.csv and detail to metrics_detailed.csv
Saved table files: metrics_table4.csv, metrics_table5.csv, metrics_table6.csv, metrics_table7.csv
